In [2]:
# === REGION + INTENSITY ONLY: 2900 version ===
# Output:
# 1) Stratified CV F1_macro
# 2) GroupKFold by region F1_macro
# 3) Holdout metrics + confusion matrix
# 4) Permutation importance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import TensorDataset, DataLoader

# -------------------- 1 LOAD DATA --------------------

CSV_PATH = "mergedData/data_feat_1500_std.csv"

df = pd.read_csv(CSV_PATH)

# target
y_str = df["class"].astype(str)

le = LabelEncoder()
y = le.fit_transform(y_str)

# region feature
region = df["position"].astype(str)

# spectral columns
wave_cols = [c for c in df.columns if str(c).startswith("Wave_")]

X_spec = df[wave_cols].to_numpy(dtype=np.float32)

# -------------------- 2 FEATURE PREPROCESSING --------------------

# one-hot region
region_ohe = OneHotEncoder(sparse_output=False)
region_encoded = region_ohe.fit_transform(region.values.reshape(-1,1))

# normalize spectra
scaler = StandardScaler()
X_spec = scaler.fit_transform(X_spec)

# combine features
X = np.hstack([region_encoded, X_spec]).astype(np.float32)

print("Feature matrix:", X.shape)

# -------------------- 3 TRAIN TEST SPLIT --------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# -------------------- 4 TORCH DEVICE --------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------------------- 5 DATASETS --------------------

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)

X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1024)

# -------------------- 6 MODEL --------------------

class MLP(nn.Module):

    def __init__(self, input_dim, n_classes):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim,256),
            nn.ReLU(),
            nn.BatchNorm1d(256),

            nn.Linear(256,128),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,n_classes)
        )

    def forward(self,x):
        return self.net(x)

model = MLP(X_train.shape[1], len(le.classes_)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=4e-4)

# -------------------- 7 TRAIN --------------------

epochs = 50

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch+1} loss {total_loss:.4f}")

# -------------------- 8 EVALUATE --------------------

model.eval()

preds = []
true = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        logits = model(xb)

        p = torch.argmax(logits, dim=1).cpu().numpy()

        preds.append(p)
        true.append(yb.numpy())

y_pred = np.concatenate(preds)
y_true = np.concatenate(true)

print("\nAccuracy:", accuracy_score(y_true, y_pred))
print("F1 macro:", f1_score(y_true, y_pred, average="macro"))

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=le.classes_))

C:\Users\Zepspel\AppData\Local\Temp\ipykernel_6788\746519278.py:15: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


Feature matrix: (61950, 1022)
Device: cuda
epoch 1 loss 147.8602
epoch 2 loss 101.6919
epoch 3 loss 63.5761
epoch 4 loss 34.8766
epoch 5 loss 17.0155
epoch 6 loss 10.0040
epoch 7 loss 6.6501
epoch 8 loss 5.2114
epoch 9 loss 5.7769
epoch 10 loss 5.6415
epoch 11 loss 5.0798
epoch 12 loss 4.8808
epoch 13 loss 3.5528
epoch 14 loss 3.0710
epoch 15 loss 2.6555
epoch 16 loss 2.5194
epoch 17 loss 2.3849
epoch 18 loss 3.3539
epoch 19 loss 4.2564
epoch 20 loss 4.5648
epoch 21 loss 3.5350
epoch 22 loss 1.9764
epoch 23 loss 0.9424
epoch 24 loss 0.7884
epoch 25 loss 0.4290
epoch 26 loss 0.8389
epoch 27 loss 0.8846
epoch 28 loss 5.0058
epoch 29 loss 8.7440
epoch 30 loss 3.5244
epoch 31 loss 1.3632
epoch 32 loss 0.6880
epoch 33 loss 0.4157
epoch 34 loss 0.6892
epoch 35 loss 0.3316
epoch 36 loss 0.2589
epoch 37 loss 0.1949
epoch 38 loss 0.4288
epoch 39 loss 3.3092
epoch 40 loss 9.2854
epoch 41 loss 4.2566
epoch 42 loss 1.3091
epoch 43 loss 0.4723
epoch 44 loss 0.1916
epoch 45 loss 0.1626
epoch 46 loss

In [6]:
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Total parameters: 304003
